# MOT20 Detection Benchmark

Benchmark YOLOv8, YOLO11, YOLO12, YOLO26, and RT-DETR on the MOT20 dataset.

The benchmark reports:
- Parameters
- Precision
- Recall
- mAP@50
- mAP@50:95
- GPU inference time
- GPU FPS
- Peak GPU memory

All models are evaluated using the same MOT20 person-detection setup.

In [ ]:
!pip install -U ultralytics

In [ ]:
import gc
import configparser
import shutil
import traceback
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO, RTDETR

MOT_ROOT = Path("/kaggle/input/datasets/malakgendy/mot20traintest/MOT20/MOT20")
WORK = Path("/kaggle/working/")
MODE = "zeroshot"
ALL_SEQS = ["MOT20-01", "MOT20-02", "MOT20-03", "MOT20-05"]
TRAIN_SEQS = ["MOT20-01", "MOT20-02", "MOT20-03"]
VAL_SEQS = ["MOT20-05"]
IMGSZ = 1280
STRIDE = 5
EPOCHS = 10
BATCH_TRAIN = 4
BATCH_VAL = 8
DEVICE = 0
SIZES = ["n", "s", "m", "l"]
FAMILIES = {
    "YOLOv8": "yolov8",
    "YOLO11": "yolo11",
    "YOLO12": "yolo12",
    "YOLO26": "yolo26",
}
RTDETR_MODELS = {
    "RT-DETR-l": "rtdetr-l.pt",
    "RT-DETR-x": "rtdetr-x.pt",
}
GT_CLASSES = {1}
TIME_H, TIME_W = 736, 1280
WARMUP, ITERS = 30, 150
TRAIN_DIR = MOT_ROOT / "train" if (MOT_ROOT / "train").exists() else MOT_ROOT
assert torch.cuda.is_available(), "Turn on a GPU in the notebook settings."
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("Mode:", MODE)

## 1. MOT20 Dataset Preparation

Convert MOT20 annotations to YOLO format and create the evaluation dataset.

The benchmark evaluates the person class only using:
- MOT20-01
- MOT20-02
- MOT20-03
- MOT20-05

Every 5th frame is used to reduce redundant frames.

In [ ]:
def build_split(seqs, split):
    img_dir = WORK / "images" / split
    lab_dir = WORK / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lab_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    n_boxes = 0
    for seq in seqs:
        sdir = TRAIN_DIR / seq
        seqinfo = sdir / "seqinfo.ini"
        if not seqinfo.exists():
            raise FileNotFoundError(f"Missing seqinfo.ini: {seqinfo}")
        ini = configparser.ConfigParser()
        ini.read(seqinfo)
        if not ini.has_section("Sequence"):
            raise ValueError(f"Invalid seqinfo.ini: {seqinfo}\nSections found: {ini.sections()}")
        W = int(ini["Sequence"]["imWidth"])
        H = int(ini["Sequence"]["imHeight"])
        ext = ini["Sequence"].get("imExt", ".jpg")
        n = int(ini["Sequence"]["seqLength"])
        gt_path = sdir / "gt" / "gt.txt"
        if not gt_path.exists():
            raise FileNotFoundError(f"Missing GT: {gt_path}")
        gt = pd.read_csv(gt_path, header=None).values
        gt = gt[(gt[:, 6] == 1) & np.isin(gt[:, 7], list(GT_CLASSES))]
        by_frame = {}
        for row in gt:
            by_frame.setdefault(int(row[0]), []).append(row[2:6])
        for f in range(1, n + 1, STRIDE):
            src = sdir / "img1" / f"{f:06d}{ext}"
            if not src.exists():
                continue
            name = f"{seq}_{f:06d}"
            dst = img_dir / f"{name}{ext}"
            if not dst.exists():
                dst.symlink_to(src)
            lines = []
            for x, y, w, h in by_frame.get(f, []):
                x1, y1 = max(0, x), max(0, y)
                x2, y2 = min(W, x + w), min(H, y + h)
                if x2 - x1 < 1 or y2 - y1 < 1:
                    continue
                lines.append(
                    f"0 {(x1 + x2) / 2 / W:.6f} "
                    f"{(y1 + y2) / 2 / H:.6f} "
                    f"{(x2 - x1) / W:.6f} "
                    f"{(y2 - y1) / H:.6f}"
                )
            (lab_dir / f"{name}.txt").write_text("\n".join(lines))
            n_boxes += len(lines)
            paths.append(str(dst))
    split_file = WORK / f"{split}.txt"
    split_file.write_text("\n".join(paths))
    print(f"{split}: {len(paths)} images, {n_boxes} person boxes")
    return split_file

def build_yaml():
    for item in ["images", "labels", "runs", "mot20.yaml", "train.txt", "val.txt"]:
        path = WORK / item
        if path.is_dir():
            shutil.rmtree(path)
        elif path.exists():
            path.unlink()
    WORK.mkdir(parents=True, exist_ok=True)
    print("Dataset root:", TRAIN_DIR)
    print("Checking sequences:")
    for seq in ALL_SEQS:
        sdir = TRAIN_DIR / seq
        print(seq, "->", sdir, "exists:", sdir.exists(), "seqinfo:", (sdir / "seqinfo.ini").exists())
    if MODE == "zeroshot":
        val = build_split(ALL_SEQS, "val")
        train = val
    else:
        train = build_split(TRAIN_SEQS, "train")
        val = build_split(VAL_SEQS, "val")
    yml = WORK / "mot20.yaml"
    yml.write_text(
        f"path: {WORK}\n"
        f"train: {train}\n"
        f"val: {val}\n"
        "names:\n"
        "  0: person\n"
    )
    return yml

yml = build_yaml()
print("Dataset YAML:", yml)

## 2. Model and Benchmark Functions

Define the functions required to:

- Load YOLO and RT-DETR models
- Evaluate detection accuracy
- Measure GPU inference time
- Calculate GPU FPS
- Measure peak GPU memory
- Fine-tune models when enabled
- Manage GPU memory

In [ ]:
def load(weights):
    return RTDETR(weights) if "rtdetr" in str(weights).lower() else YOLO(weights)

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

@torch.inference_mode()
def time_model(weights):
    cleanup()
    torch.cuda.reset_peak_memory_stats()
    m = load(weights)
    try:
        m.fuse()
    except Exception:
        pass
    net = m.model.to(f"cuda:{DEVICE}").half().eval()
    params = sum(p.numel() for p in net.parameters()) / 1e6
    x = torch.randn(1, 3, TIME_H, TIME_W, device=f"cuda:{DEVICE}").half()
    for _ in range(WARMUP):
        net(x)
    torch.cuda.synchronize()
    start = torch.cuda.Event(True)
    end = torch.cuda.Event(True)
    start.record()
    for _ in range(ITERS):
        net(x)
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end) / ITERS
    fps = 1000 / ms
    peak = torch.cuda.max_memory_allocated() / 1024 ** 3
    del net, m, x
    cleanup()
    return params, ms, fps, peak

def accuracy(weights, yml):
    m = load(weights)
    r = m.val(
        data=str(yml),
        imgsz=IMGSZ,
        batch=BATCH_VAL,
        conf=0.001,
        iou=0.6,
        classes=[0],
        half=True,
        device=DEVICE,
        plots=False,
        verbose=False,
        workers=2,
        project=str(WORK / "runs"),
        name="val",
        exist_ok=True
    )
    result = (
        float(r.box.mp),
        float(r.box.mr),
        float(r.box.map50),
        float(r.box.map)
    )
    del m, r
    cleanup()
    return result

def finetune(weights, tag, yml):
    m = load(weights)
    m.train(
        data=str(yml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH_TRAIN,
        device=DEVICE,
        workers=2,
        plots=False,
        val=False,
        project=str(WORK / "runs"),
        name=tag,
        exist_ok=True
    )
    last = WORK / "runs" / tag / "weights" / "last.pt"
    del m
    cleanup()
    return last

def model_list():
    items = []
    for display_name, prefix in FAMILIES.items():
        for size in SIZES:
            items.append((f"{display_name}{size}", f"{prefix}{size}.pt"))
    items += list(RTDETR_MODELS.items())
    return items

## 3. Run the Benchmark

Run the detection benchmark for all selected models using the same evaluation settings.

The results are saved after each completed model so the benchmark can resume if the notebook is interrupted.

In [ ]:
csv_path = WORK / "mot20_benchmark.csv"
rows = pd.read_csv(csv_path).to_dict("records") if csv_path.exists() else []
done = {r["Detector"] for r in rows}

for name, weight in model_list():
    if name in done:
        continue
    print(f"\n=== {name} ({weight}) ===")
    try:
        weights = finetune(weight, name, yml) if MODE == "finetune" else weight
        precision, recall, map50, map5095 = accuracy(weights, yml)
        params, ms, fps, peak = time_model(weights)
        row = {
            "Detector": name,
            "Params (M)": round(params, 2),
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "mAP@50": round(map50, 4),
            "mAP@50:95": round(map5095, 4),
            "GPU inference (ms)": round(ms, 2),
            "GPU FPS": round(fps, 2),
            "Peak GPU (GB)": round(peak, 2)
        }
        rows.append(row)
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        print(row)
    except Exception:
        print(f"!! {name} failed")
        traceback.print_exc()
        cleanup()

print("\nBenchmark finished.")

## 4. Benchmark Results

Sort the models by mAP@50:95 and generate the final comparison.

The results are saved as:
- CSV
- Markdown table
- PNG comparison table

In [ ]:
def render_png(df, out):
    import matplotlib.pyplot as plt
    heads = [
        "Detector",
        "Params\n(M)",
        "Precision\n↑",
        "Recall\n↑",
        "mAP@50\n↑",
        "mAP@50:95\n↑",
        "GPU\ninference\n(ms) ↓",
        "GPU\nFPS ↑",
        "Peak\nGPU\n(GB) ↓"
    ]
    fmt = [
        "{}",
        "{:.2f}",
        "{:.4f}",
        "{:.4f}",
        "{:.4f}",
        "{:.4f}",
        "{:.2f}",
        "{:.2f}",
        "{:.2f}"
    ]
    text = [[f.format(v) for f, v in zip(fmt, row)] for row in df.values]
    fig, ax = plt.subplots(figsize=(11, 0.45 * len(df) + 1.8))
    ax.axis("off")
    table = ax.table(
        cellText=text,
        colLabels=heads,
        loc="center",
        cellLoc="right"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)
    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("#d0d7de")
        if r == 0:
            cell.set_facecolor("#ffffff")
            cell.set_text_props(weight="bold")
            cell.set_height(cell.get_height() * 1.8)
        elif r % 2 == 0:
            cell.set_facecolor("#f6f8fa")
        if c == 0:
            cell.set_text_props(ha="left")
            cell._loc = "left"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()

df = (
    pd.DataFrame(rows)
    .sort_values("mAP@50:95", ascending=False)
    .reset_index(drop=True)
)
df.to_csv(WORK / "mot20_benchmark.csv", index=False)
(WORK / "mot20_benchmark.md").write_text(df.to_markdown(index=False))
print(df.to_string(index=False))
render_png(df, WORK / "mot20_benchmark.png")